## Arctic Shadow Tracker - AIS Data

In [1]:
import requests
import pandas as pd
import folium
from datetime import datetime
import yaml
import os

In [2]:
from src.mmsi_country_reference import MMSI_COUNTRY_MAP
from src.ais_ship_types import get_ship_type

In [3]:
# Load credentials - try environment variables first (GitHub), then config.yaml (local)
def load_credentials():
    # GitHub Actions: use environment variables
    client_secret = os.getenv('BARENTSWATCH_CLIENT_SECRET')
    if client_secret:
        print("Using credentials from environment variables")
        return {
            'client_id': 'henrikformoe@gmail.com:ArcticShadowTrackerAIS',
            'client_secret': client_secret
        }
    
    # Local development: use config.yaml
    try:
        with open('config.yaml', 'r') as file:
            config = yaml.safe_load(file)
            print("Using credentials from config.yaml")
            return {
                'client_id': config['barentswatch']['client_id'],
                'client_secret': config['barentswatch']['client_secret']
            }
    except FileNotFoundError:
        raise Exception("No config.yaml found and no environment variables set")

credentials = load_credentials()
CLIENT_ID = credentials['client_id']
CLIENT_SECRET = credentials['client_secret']

# BarentsWatch API endpoints
TOKEN_URL = "https://id.barentswatch.no/connect/token"
AIS_URL = "https://www.barentswatch.no/bwapi/v1/geodata/ais"

def get_barentswatch_token(client_id, client_secret):
    response = requests.post(
        'https://id.barentswatch.no/connect/token',
        data={
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'ais',
            'grant_type': 'client_credentials'
        }
    )
    
    if response.status_code == 200:
        return response.json()['access_token']
    else:
        raise Exception(f"Failed to get token: {response.json()}")

token = get_barentswatch_token(CLIENT_ID, CLIENT_SECRET)
print("Got token:", token[:20] + "...")

Using credentials from config.yaml
Got token: eyJhbGciOiJSUzI1NiIs...


###

In [4]:
# Arctic region definition
ARCTIC_REGION = {
    'lat_min': 65.0,   # Above Lofoten
    'lat_max': 82.0,   # All of Svalbard
    'lon_min': 0.0,    # Western Norway
    'lon_max': 40.0    # Past Kola Peninsula
}

In [5]:
# Fetch AIS data
print("Fetching AIS data...")
headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/json'}
url = "https://live.ais.barentswatch.no/v1/latest/combined"

response = requests.get(url, headers=headers, timeout=30)
all_vessels = response.json()

print(f"Total vessels received: {len(all_vessels)}")

# Filter for Arctic Russian/Chinese vessels
target_vessels = []

for v in all_vessels:
    lat = v.get('latitude', 0)
    lon = v.get('longitude', 0)
    
    if not (ARCTIC_REGION['lat_min'] <= lat <= ARCTIC_REGION['lat_max'] and 
            ARCTIC_REGION['lon_min'] <= lon <= ARCTIC_REGION['lon_max']):
        continue
    
    mmsi = str(v.get('mmsi', ''))
    mmsi_prefix = mmsi[:3]
    
    # Use existing mapping
    country = MMSI_COUNTRY_MAP.get(mmsi_prefix, 'Unknown')
    
    if country in ['Russia', 'China']:
        target_vessels.append({
            'timestamp': datetime.now().isoformat(),
            'mmsi': mmsi,
            'name': v.get('name', 'Unknown'),
            'country': country,
            'latitude': lat,
            'longitude': lon,
            'speed': v.get('speedOverGround') or 0,
            'course': v.get('courseOverGround') or 0,
            'ship_type': get_ship_type(v.get('shipType', 0)),  # Use existing function
            'ship_type_code': v.get('shipType', 0)
        })

print(f"\n Found {len(target_vessels)} Russian/Chinese vessels in Arctic")
print(f"   - Russian: {sum(1 for v in target_vessels if v['country'] == 'Russia')}")
print(f"   - Chinese: {sum(1 for v in target_vessels if v['country'] == 'China')}")

Fetching AIS data...
Total vessels received: 4142

 Found 31 Russian/Chinese vessels in Arctic
   - Russian: 29
   - Chinese: 2


In [6]:
# Display summary
df = pd.DataFrame(target_vessels)
print("\nVessel breakdown by type:")
print(df.groupby(['country', 'ship_type']).size())
df.head(10)


Vessel breakdown by type:
country  ship_type                    
China    Cargo                             1
         Fishing                           1
Russia   Cargo                             4
         Fishing                          21
         Other, no additional info         1
         Passenger, no additional info     1
         Tanker                            1
         Tanker, no additional info        1
dtype: int64


,timestamp,mmsi,name,country,latitude,longitude,speed,course,ship_type,ship_type_code
0,2025-10-03T13:06:01.992282,273437390,ALLIKSAARE,Russia,70.950000,32.081667,0.0,0.0,Fishing,30
1,2025-10-03T13:06:01.992319,273213010,OBELIAI,Russia,74.639287,21.029257,3.8,1.4,"Other, no additional info",99
2,2025-10-03T13:06:01.992325,273213180,VALENTIN MANTUROV,Russia,73.983333,15.781667,8.0,197.0,Fishing,30
3,2025-10-03T13:06:01.992348,273516500,VOYKOVO,Russia,73.687258,32.022873,1.0,191.3,Fishing,30
4,2025-10-03T13:06:01.992391,273291510,NORVEZHSKOE MORE,Russia,74.234875,16.259843,5.1,156.7,Fishing,30
5,2025-10-03T13:06:01.992405,273217010,BOOTES,Russia,74.486613,19.266530,0.0,194.0,Fishing,30
6,2025-10-03T13:06:01.992433,273897000,MYS SLEPIKOVSKOGO,Russia,73.320203,15.078602,7.3,288.5,Fishing,30
7,2025-10-03T13:06:01.992482,273522400,ZAKHAR SOROKIN,Russia,70.647500,19.173883,4.1,241.8,Fishing,30
8,2025-10-03T13:06:01.992484,273447010,ALFERAS,Russia,74.794892,18.206033,3.8,61.4,Fishing,30
9,2025-10-03T13:06:01.992507,414957000,GUOJIANENGYUAN603,China,72.146667,27.213333,54.0,352.0,Cargo,70


In [7]:
# Create map with vessels and grid boundary
m = folium.Map(location=[75, 20], zoom_start=5)
colors = {'Russia': 'red', 'China': 'orange'}

# Add grid boundary rectangle (outer bounds only)
folium.Rectangle(
    bounds=[[65, 0], [82, 40]],  # lat_min, lon_min to lat_max, lon_max
    color='blue',
    fill=False,
    weight=2,
    opacity=0.5,
    popup='Coverage Area: 65-82°N, 0-40°E'
).add_to(m)

# Add vessel markers
for v in target_vessels:
    speed = v.get('speed') or 0
    course = v.get('course') or 0
    
    folium.CircleMarker(
        location=[v['latitude'], v['longitude']],
        radius=8,
        popup=folium.Popup(
            f"<b>{v['name']}</b><br>"
            f"Country: {v['country']}<br>"
            f"Type: {v['ship_type']}<br>"
            f"MMSI: {v['mmsi']}<br>"
            f"Speed: {speed:.1f} knots<br>"
            f"Course: {course:.1f}°",
            max_width=300
        ),
        color=colors[v['country']],
        fill=True,
        fillOpacity=0.7
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; top: 10px; right: 10px; z-index:9999; background-color:white; 
            padding: 10px; border:2px solid grey; border-radius: 5px;">
    <p style="margin:0;"><b>Arctic Vessels</b></p>
    <p style="margin:0;"><span style="color:red;">●</span> Russia ({rus})</p>
    <p style="margin:0;"><span style="color:orange;">●</span> China ({chn})</p>
    <p style="margin:0; margin-top:5px; color:blue;">▭ Coverage Area</p>
</div>
'''.format(
    rus=sum(1 for v in target_vessels if v['country'] == 'Russia'),
    chn=sum(1 for v in target_vessels if v['country'] == 'China')
)

m.get_root().html.add_child(folium.Element(legend_html))

# Save to outputs folder
os.makedirs('outputs', exist_ok=True)
map_file = f'outputs/arctic_map_{datetime.now().strftime("%Y%m%d_%H%M")}.html'
# m.save(map_file)
print(f"\n✓ Map saved: {map_file}")
m


✓ Map saved: outputs/arctic_map_20251003_1306.html


# Save data
output = {
    'timestamp': datetime.now().isoformat(),
    'vessel_count': len(target_vessels),
    'by_country': {
        'Russia': sum(1 for v in target_vessels if v['country'] == 'Russia'),
        'China': sum(1 for v in target_vessels if v['country'] == 'China')
    },
    'vessels': target_vessels
}

json_file = f'arctic_vessels_{datetime.now().strftime("%Y%m%d_%H%M")}.json'
with open(json_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f"✓ Data saved: {json_file}")
print(f"\nSnapshot complete: {len(target_vessels)} vessels tracked")